In [1]:
import os
from os.path import join, basename
from multiprocessing import Pool, cpu_count

import numpy as np
import tifffile as tiff
from sklearn.metrics import confusion_matrix
from scipy.sparse import lil_array

from ctc_metrics.utils.filesystem import parse_directories, read_tracking_file,\
    parse_masks
# from ctc_metrics.utils.representations import match as match_tracks, \
#     count_acyclic_graph_correction_operations, merge_tracks
from ctc_metrics.metrics import (
    valid, det, seg, tra, ct, tf, bc, cca, mota, hota, idf1, chota, mtml, faf,
    op_ctb, op_csb, bio, op_clb, lnk
)

def match_tracks(
        ref_path: str,
        comp_path: str
):
    """
    Matches the labels of the masks from the reference and computed result path.
    A label is matched if the intersection of a computed and reference mask is
    greater than 50% of the area of the reference mask.

    Args:
        ref_path: Path to the reference mask.
        comp_path: Path to the computed mask.

    Returns:
        A tuple of five numpy arrays. 
        - The first array contains the existing labels in the reference mask. 
        - The second array contains the existing labels in the computed mask. 
        - The third array contains the matched labels in the referenced mask.
        - The fourth array contains the corresponding matched labels in the computed mask. 
        - The fiftharray contains the intersection over union (IoU) for each matched label pair.
    """
    # Read the input data
    if comp_path is None:
        map_ref = tiff.imread(ref_path)
        labels_ref = np.unique(map_ref)
        labels_ref = labels_ref[labels_ref != 0]
        return labels_ref.tolist(), [], [], [], []
    if ref_path is None:
        # For trivial cases where only one mask should be analysed
        ref_path = comp_path
    if os.path.exists(ref_path):
        # No slices
        map_ref = tiff.imread(ref_path)
        map_com = tiff.imread(comp_path)
    else:
        # Slices in 3D dataset segmentation data
        dirname, base = os.path.dirname(ref_path), os.path.basename(ref_path)
        base = base.replace(".tif", "")
        slice_files = [x for x in os.listdir(dirname) if
                 x.endswith(".tif") and x.startswith(base)]
        slice_files = sorted(slice_files)
        map_ref = [tiff.imread(os.path.join(dirname, x)) for x in slice_files]
        map_ref = np.stack(map_ref, axis=0)
        slices = [x.replace(base+"_", "").replace(".tif", "")
                  for x in slice_files]
        slices = [int(x) for x in slices]
        map_com = tiff.imread(comp_path)
        _map_com = [map_com[x] for x in slices]
        map_com = np.stack(_map_com, axis=0)
    # Get the labels of the two masks (including background label 0)
    labels_ref, labels_comp = np.unique(map_ref), np.unique(map_com)
    if ref_path == comp_path:
        # For trivial cases where only one mask should be analysed
        iou = np.ones(len(labels_ref))
        labels_ref = labels_ref[labels_ref != 0]
        labels_comp = labels_comp[labels_comp != 0]
        return labels_ref.tolist(), labels_comp.tolist(), labels_ref.tolist(),\
            labels_comp.tolist(), iou.tolist()
    # Add offset to separate the labels of the two masks
    offset = int(np.max(labels_ref) + 1)
    map_com += offset
    # Compute the confusion matrix
    cm = confusion_matrix(map_ref.flatten(), map_com.flatten())
    sum_ref = np.sum(cm, axis=1, keepdims=True)
    sum_comp = np.sum(cm, axis=0, keepdims=True)
    # Compute the intersection over reference
    intersection_over_ref = cm / np.maximum(sum_ref, 1)
    # Compute the intersection over union (relevant to calculate SEG)
    intersection_over_union = cm / np.maximum(sum_ref + sum_comp - cm, 1)
    # Remove the background label and redundant parts of the matrix
    intersection_over_ref = \
        intersection_over_ref[1:len(labels_ref), 1 + len(labels_ref):]
    intersection_over_union = \
        intersection_over_union[1:len(labels_ref), 1 + len(labels_ref):]
    # Find matches according to AOGM (min 50% of ref needs to be covered)
    intersection_over_ref[intersection_over_ref <= 0.5] = 0
    intersection_over_ref[intersection_over_ref > 0.5] = 1
    # Create mapping between reference and computed labels
    rows, cols = np.nonzero(intersection_over_ref)
    labels_ref = labels_ref[1:]
    labels_comp = labels_comp[1:]
    mapped_ref = labels_ref[rows].tolist()
    mapped_comp = labels_comp[cols].tolist()
    iou = intersection_over_union[rows, cols].tolist()
    assert np.unique(mapped_ref).size == len(mapped_ref), \
        f"Reference node assigned to multiple computed nodes! " \
        f"{ref_path} {labels_ref, labels_comp, mapped_ref, mapped_comp}"
    labels_ref = labels_ref.tolist()
    labels_comp = labels_comp.tolist()
    return labels_ref, labels_comp, mapped_ref, mapped_comp, iou


In [ ]:
root = r'D:\dataset\cell-benchmark\test_dataset_ctc\train\BF-C2DL-HSC'
res = join(root, '01_RES')
gt = join(root, '01_GT')
trajectory_data = True
segmentation_data= True
threads = 1

def match_computed_to_reference_masks(
        ref_masks: list,
        comp_masks: list,
        threads: int = 0,
):
    """
    Matches computed masks to reference masks.

    Args:
        ref_masks: The reference masks. A list of paths to the reference masks.
        comp_masks: The computed masks. A list of paths to the computed masks.
        threads: The number of threads to use. If 0, the number of threads
            is set to the number of available CPUs.

    Returns:
        The results stored in a dictionary. The dictionary contains the
        following keys:
            - labels_ref: The reference labels. A list of lists containing
                the labels of the reference masks.
            - labels_comp: The computed labels. A list of lists containing
                the labels of the computed masks.
            - mapped_ref: The mapped reference labels. A list of lists
                containing the mapped labels of the reference masks.
            - mapped_comp: The mapped computed labels. A list of lists
                containing the mapped labels of the computed masks.
            - ious: The intersection over union values. A list of lists
                containing the intersection over union values between mapped
                reference and computed masks.
    """
    labels_ref, labels_comp, mapped_ref, mapped_comp, ious = [], [], [], [], []
    if threads != 1:
        if threads == 0:
            threads = cpu_count()
        with Pool(threads) as p:
            matches = p.starmap(match_tracks, zip(ref_masks, comp_masks))
    else:
        matches = [match_tracks(*x) for x in zip(ref_masks, comp_masks)]
    for match in matches:
        labels_ref.append(match[0])
        labels_comp.append(match[1])
        mapped_ref.append(match[2])
        mapped_comp.append(match[3])
        ious.append(match[4])
    return {
        "labels_ref": labels_ref,
        "labels_comp": labels_comp,
        "mapped_ref": mapped_ref,
        "mapped_comp": mapped_comp,
        "ious": ious
    }

comp_tracks = read_tracking_file(join(res, "res_track.txt"))
ref_tracks = read_tracking_file(join(gt, "TRA", "man_track.txt"))
comp_masks = parse_masks(res)
ref_tra_masks = parse_masks(join(gt, "TRA"))
assert len(ref_tra_masks) > 0, f"{gt}: Ground truth masks is 0!)"
assert len(ref_tra_masks) == len(comp_masks), (
    f"{res}: Number of result masks ({len(comp_masks)}) unequal to "
    f"the number of ground truth masks ({len(ref_tra_masks)})!)")
# Match golden truth tracking masks to result masks
traj = {}
is_valid = 1
if trajectory_data:
    traj = match_computed_to_reference_masks(
        ref_tra_masks, comp_masks, threads=threads)
    is_valid = valid(comp_masks, comp_tracks, traj["labels_comp"])
# Match golden truth segmentation masks to result masks
segm = {}
if segmentation_data:
    ref_seg_masks = parse_masks(join(gt, "SEG"))
    _res_masks = [
        comp_masks[int(basename(x).replace(
            "man_seg", "").replace(".tif", "").replace("_", ""))]
        for x in ref_seg_masks
    ]
    segm = match_computed_to_reference_masks(
        ref_seg_masks, _res_masks, threads=threads)
# return comp_tracks, ref_tracks, traj, segm, comp_masks, is_valid